# EUDR DeepLabV3 Training — Kaggle GPU

Single-image segmentation baseline (M1a). Trains on 2020 Sentinel-2 composites with Hansen GFC labels.

**Accelerator:** T4 x2 — enable via *Settings → Accelerator → GPU T4 x2*  
**⚠️ P100 is NOT supported** — current PyTorch requires sm_70+; P100 is sm_60. Use T4 only.

### Kaggle Datasets to attach:
- `eudr-satellite` — contains `2020_baseline/`, `2024_current/`, `hansen_labels/`, `farms_osm.csv`

### Kaggle Secrets required:
- `GEE_PROJECT_ID`
- `GITHUB_TOKEN` — personal access token with `repo` scope (needed to clone private repo)

In [ ]:
import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    cc = props.major * 10 + props.minor
    note = "OK" if cc >= 70 else f"INCOMPATIBLE (sm_{cc} < sm_70) — switch accelerator to T4"
    print(f"  GPU {i}: {props.name} | {props.total_memory / 1e9:.1f} GB | sm_{cc} [{note}]")
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Enable T4 x2 via Settings → Accelerator.")

In [ ]:
# Clone repo — uses GITHUB_TOKEN secret for private repo access
import subprocess, os

BRANCH   = "Tessera-head"
WORK_DIR = "/kaggle/working/eudr"

if not os.path.exists(WORK_DIR):
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        repo_url = f"https://{token}@github.com/rajul-kk/EUDR-compliance.git"
    except Exception:
        repo_url = "https://github.com/rajul-kk/EUDR-compliance.git"
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, repo_url, WORK_DIR],
        check=True,
    )
else:
    subprocess.run(["git", "-C", WORK_DIR, "pull"], check=True)

os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
import hashlib, pathlib, subprocess

_req = pathlib.Path("requirements.txt").read_bytes()
_req_hash = hashlib.md5(_req).hexdigest()
_marker = pathlib.Path("/kaggle/working/.pip_hash")

if not _marker.exists() or _marker.read_text() != _req_hash:
    subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
    _marker.write_text(_req_hash)
else:
    print("pip cache hit — skipping install")

In [ ]:
from kaggle_secrets import UserSecretsClient
import os
secrets = UserSecretsClient()
os.environ["GEE_PROJECT_ID"] = secrets.get_secret("GEE_PROJECT_ID")
print("Credentials loaded.")

In [ ]:
DATA = "/kaggle/input/eudr-satellite"
T1_DIR    = f"{DATA}/2020_baseline"
LABEL_DIR = f"{DATA}/hansen_labels"

import os
os.makedirs("models", exist_ok=True)

for p in [T1_DIR, LABEL_DIR]:
    print(f"[{'OK' if os.path.exists(p) else 'MISSING'}] {p}")

In [ ]:
# Train DeepLabV3 (M1a) — single-image segmentation on 2020 baseline
# DataParallel activates automatically when GPU count > 1
# batch-size 8 = 4 per T4 GPU
!python src/ML_farm_net.py \
    --raw-dir   {T1_DIR} \
    --mask-dir  {LABEL_DIR} \
    --output-model-path /kaggle/working/models/farm_deeplab.pth \
    --epochs        15 \
    --batch-size     8 \
    --learning-rate 1e-4 \
    --seed          42

In [ ]:
import torch
ckpt = torch.load("/kaggle/working/models/farm_deeplab.pth", map_location="cpu")
print("Checkpoint keys:", list(ckpt.keys()))

In [ ]:
!zip /kaggle/working/farm_deeplab.zip /kaggle/working/models/farm_deeplab.pth
print("Download farm_deeplab.zip from the Output panel.")